<a href="https://colab.research.google.com/github/Arti1423/practice_program_2022/blob/main/news_api_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import csv
import pandas as pd
from datetime import datetime
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import spacy

In [ ]:
# Initialize NLTK's VADER sentiment analyzer
nltk.download('vader_lexicon')
sid = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [ ]:
# Load spaCy's English NER model
nlp = spacy.load('en_core_web_sm')  # Make sure you have downloaded the model with: python -m spacy download en_core_web_sm

# Function to extract location using spaCy's NER
def extract_location(content):
    doc = nlp(content)
    locations = [ent.text for ent in doc.ents if ent.label_ == 'GPE']  # GPE = Geopolitical Entity (countries, cities, states)
    return locations[0] if locations else "Unknown"

In [ ]:
# Function to determine disaster type based on keywords
def determine_disaster_type(content, disaster):
    content_lower = content.lower()
    if 'flood' in content_lower:
        return 'Flood'
    elif 'hurricane' in content_lower or 'cyclone' in content_lower:
        return 'Hurricane/Cyclone'
    elif 'earthquake' in content_lower:
        return 'Earthquake'
    elif 'fire' in content_lower or 'wildfire' in content_lower:
        return 'Fire'
    elif 'tsunami' in content_lower:
        return 'Tsunami'
    else:
      if disaster == "Hurricane" or disaster == "Cyclone":
        return 'Hurricane/Cyclone'
      else:
        return disaster

In [ ]:
# Function to determine urgency based on keywords
def determine_urgency(content):
    urgency_keywords = ['urgent', 'immediate', 'critical', 'severe', 'emergency']
    for keyword in urgency_keywords:
        if keyword in content.lower():
            return 'High'
    return 'Medium'  # Default urgency level

In [ ]:
# News API configuration
api_key = 'b1e3e12e9d334925a1d4f2b390c29102'  # Replace with your News API key
url = 'https://newsapi.org/v2/everything'

def get_disaster_news(q):
  # Query parameters for the News API
  params = {
      'q': f'"{q}"',  # Keywords to search for
      'language': 'en',  # Language of the news
      'sortBy': 'publishedAt',  # Sort by published date
      'apiKey': api_key,
      'searchIn': 'title'
  }

  # Make a request to the News API
  response = requests.get(url, params=params).json()
  articles = response.get('articles', [])

  return articles


In [ ]:
# Prepare data for CSV
data = []

disaster_types = ["Flood", "Hurricane", "Earthquake", "Cyclone", "Fire", "Tsunami"]
for disaster in disaster_types:
  articles = get_disaster_news(disaster)

  for article in articles:
        timestamp = article['publishedAt']
        source = article['source']['name']
        content = article['description'] or article['title']
        title = article['title']
        _content = article['content']


        if not content or content == "[Removed]":
          continue

        # Determine disaster type from content
        disaster_type = determine_disaster_type(content, disaster)

        # Extract location from content
        location = extract_location(_content + content + title)

        # Determine urgency based on content
        urgency = determine_urgency(content)

        # Sentiment analysis on content
        sentiment_score = sid.polarity_scores(content)['compound']

        # Append row data
        data.append([timestamp, source, location, content, title, disaster_type, urgency, sentiment_score])

In [ ]:
df = pd.DataFrame(data, columns=['timestamp', 'source', 'location', 'content', 'title', 'disaster_type', 'urgency', 'sentiment_score'])

# Save to CSV
df.to_csv('disaster_news_data_v2.csv', index=True)
print(f"Data saved to disaster_news_data.csv with {len(data)} rows")


Data saved to disaster_news_data.csv with 518 rows


# New section